Single Geodatabase

In [ ]:
import arcpy
import os

gdb_path = r"E:\STUDENT\AUCE\Test\Doiwala_Feature Extraction.gdb" 

def main():
    try:
        arcpy.env.workspace = gdb_path
        data_list = []
        
        print("Reading Geodatabase... please wait.")

        datasets = arcpy.ListDatasets(feature_type='feature')
        
        if datasets:
            for ds in datasets:
                arcpy.env.workspace = os.path.join(gdb_path, ds)
                fcs = arcpy.ListFeatureClasses()
                
                for fc in fcs:
                    count = int(arcpy.GetCount_management(fc).getOutput(0))
                    data_list.append((ds, fc, count))
        
        arcpy.env.workspace = gdb_path
        standalone_fcs = arcpy.ListFeatureClasses()
        
        if standalone_fcs:
            for fc in standalone_fcs:
                count = int(arcpy.GetCount_management(fc).getOutput(0))
                data_list.append(("None", fc, count))

        data_list.sort(key=lambda x: (x[0] == "None", x[0], x[1]))

        header = f"{'Feature Dataset':<30} | {'Feature Class':<40} | {'Count':<10}"
        print("\n" + header)
        print("-" * len(header))

        for row in data_list:
            dataset_name = row[0]
            fc_name = row[1]
            fc_count = str(row[2])
            
            print(f"{dataset_name:<30} | {fc_name:<40} | {fc_count:<10}")
            print("-" * len(header))

    except Exception as e:
        print(f"An error occurred: {e}")

if __name__ == "__main__":
    main()

Multiple Geodatabases

In [ ]:
import arcpy
import os
import pandas as pd

gdb_folder = r"E:\STUDENT\AUCE\Test"
output_folder=r"E:\STUDENT\AUCE\Test"
def main():
    all_data = []
    
    gdbs = [f for f in os.listdir(gdb_folder) if f.endswith(".gdb")]
    
    if not gdbs:
        print("No Geodatabases found in the specified directory.")
        return

    print(f"Processing {len(gdbs)} Geodatabases...")

    for gdb_name in gdbs:
        city_name = gdb_name.split('_')[0]
        full_path = os.path.join(gdb_folder, gdb_name)
        
        arcpy.env.workspace = full_path
        
        datasets = arcpy.ListDatasets(feature_type='feature') or []
        for ds in datasets:
            ds_path = os.path.join(full_path, ds)
            arcpy.env.workspace = ds_path
            fcs = arcpy.ListFeatureClasses() or []
            for fc in fcs:
                count = int(arcpy.GetCount_management(fc).getOutput(0))
                all_data.append({
                    'Feature Dataset': ds, 
                    'Feature Class': fc, 
                    'City': city_name, 
                    'Count': count
                })

        arcpy.env.workspace = full_path
        standalone_fcs = arcpy.ListFeatureClasses() or []
        for fc in standalone_fcs:
            count = int(arcpy.GetCount_management(fc).getOutput(0))
            all_data.append({
                'Feature Dataset': 'None', 
                'Feature Class': fc, 
                'City': city_name, 
                'Count': count
            })

    df = pd.DataFrame(all_data)

    pivot_df = df.pivot_table(
        index=['Feature Dataset', 'Feature Class'], 
        columns='City', 
        values='Count', 
        fill_value=0
    ).reset_index()

    pivot_df['sort_rank'] = pivot_df['Feature Dataset'].apply(lambda x: 1 if x == 'None' else 0)
    
    pivot_df = pivot_df.sort_values(by=['sort_rank', 'Feature Dataset', 'Feature Class'])
    
    pivot_df = pivot_df.drop(columns=['sort_rank'])

    table_string = pivot_df.to_string(index=False)
    lines = table_string.split('\n')
    
    header = lines[0]
    underline = "-" * len(header)

    print("\n" + header)
    print(underline)
    for row in lines[1:]:
        print(row)

    output_path = os.path.join(output_folder, "Consolidated_Feature_Counts.csv")
    pivot_df.to_csv(output_path, index=False)
    print(f"\nResults exported to: {output_path}")

if __name__ == "__main__":
    main()